# Slow-starting top-20 picks: do any become stars?

**Why:** in the 2026 trade period Port Adelaide got Luke Pedlar from Adelaide.
Pedlar was pick 11 in the 2020 national draft, but injuries kept interrupting
his career at the Crows: an ankle injury and a shoulder reconstruction in 2021,
osteitis pubis in 2022, and two hamstring injuries in 2026. He played 49 games
in six seasons (2-3-21-4-9-10), with a breakout 21-game, 25-goal season in
2023. Port gave up very little: Adelaide sent Pedlar and pick 34 for pick 32
and a 2027 fourth-round pick.

**Question:** of top-20 picks who started their careers this slowly, how many
went on to become stars?

**Definitions:**

- **Population:** national picks 1–20. Rates use the 2005–2018 drafts, so every
  player has had at least 8 seasons. The list of stars uses 2005–2021.
- **Slow start:** under 50 AFL games in the first 5 seasons after the draft.
  Pedlar played 39.
- **Star season:** a season in the All-Australian squad or the Brownlow top 40,
  the "either" definition from
  [star_curve_definitions.md](../docs/star_curve_definitions.md). Counted over
  the whole career to 2026, so late bloomers are included.

**Limitation:** our data has games played, not injuries. A slow start can mean
injury, but it can also mean a player wasn't ready or wasn't good enough. Key
position players in particular often take 4–5 years to develop. This notebook
is exploratory.

In [1]:
import pandas as pd

from afl.afltables import all_season_games
from afl.datasets import draft_outcomes
from afl.stars import all_australian_squad, star_season_counts

pd.set_option("display.width", 220)

seasons = all_season_games(2006, 2026)
squad = all_australian_squad(2006, 2026)
picks = draft_outcomes(2005, 2021, windows=(5,))
top20 = picks[picks["pick"] <= 20].copy()
print(f"National picks 1-20, 2005-2021: {len(top20)}")

National picks 1-20, 2005-2021: 340


In [2]:
# Star seasons over the whole career, and games per season
top20 = top20.join(star_season_counts(top20, seasons, squad, years=None)[["squad", "either40"]])
games = seasons.groupby(["afltables_url", "season"])["games"].sum()


def games_in(row, first, last):
    """Games in seasons draft_year+first to draft_year+last."""
    player = games.get(row["afltables_url"], pd.Series(dtype=int))
    return int(player[(player.index >= row["draft_year"] + first) &
                      (player.index <= row["draft_year"] + last)].sum())


top20["first_5"] = top20.apply(
    lambda r: "-".join(str(games_in(r, i, i)) for i in range(1, 6)), axis=1)
top20["games_after_5"] = top20.apply(lambda r: games_in(r, 6, 100), axis=1)
top20["career_games"] = top20.apply(lambda r: games_in(r, 1, 100), axis=1)
top20["slow"] = top20["games_5"] < 50

COLS = ["draft_year", "pick", "player", "club", "height_cm", "first_5",
        "games_after_5", "career_games", "squad", "either40"]
top20.loc[top20["player"] == "Luke Pedlar", COLS]

,draft_year,pick,player,club,height_cm,first_5,games_after_5,career_games,squad,either40
1103,2020,11,Luke Pedlar,Adelaide,183.0,2-3-21-4-9,10,49,0,0


## 1. Star rates: slow starters against everyone else (2005–2018 drafts)

"Played on" means the player played at least one AFL game after his fifth
season, as Pedlar will at Port.

In [3]:
done = top20[top20["draft_year"] <= 2018]
slow = done[done["slow"]]
played_on = slow[slow["games_after_5"] > 0]
groups = {
    "All top-20 picks": done,
    "Under 50 games in 5 seasons": slow,
    "  ...and played on after year 5": played_on,
    "  ...and gone by year 6": slow.drop(played_on.index),
    "50+ games in 5 seasons": done[~done["slow"]],
}
pd.DataFrame([{
    "group": name,
    "n": len(g),
    "star season %": round(100 * (g["either40"] >= 1).mean()),
    "2+ star seasons %": round(100 * (g["either40"] >= 2).mean()),
    "AA squad %": round(100 * (g["squad"] >= 1).mean()),
    "median career games": int(g["career_games"].median()),
} for name, g in groups.items()]).set_index("group")

,n,star season %,2+ star seasons %,AA squad %,median career games
group,,,,,
All top-20 picks,280,40,29,34,152
Under 50 games in 5 seasons,90,4,2,3,34
...and played on after year 5,52,8,4,6,63
...and gone by year 6,38,0,0,0,21
50+ games in 5 seasons,190,57,42,49,187


## 2. The slow starters who had a star season (2005–2021 drafts)

In [4]:
top20[top20["slow"] & (top20["either40"] >= 1)].sort_values(
    "either40", ascending=False)[COLS].reset_index(drop=True)

,draft_year,pick,player,club,height_cm,first_5,games_after_5,career_games,squad,either40
0,2015,10,Harry McKay,Carlton,200.0,0-2-13-20-13,115,163,2,3
1,2008,13,Tom Lynch,St Kilda,191.0,0-1-5-6-17,135,164,2,2
2,2007,15,Robbie Tarrant,North Melbourne,196.0,0-0-2-7-16,169,194,1,1
3,2016,9,Will Brodie,Gold Coast,189.0,3-8-8-1-5,31,56,0,1


## 3. Tall and mid-sized slow starters

Pedlar is 183cm. Tall players (191cm+) often need longer to develop, so a slow
start means something different for them.

In [5]:
played_on = played_on.assign(build=lambda d: d["height_cm"].ge(191).map(
    {True: "Tall (191cm+)", False: "Mid-sized (under 191cm)"}))
played_on.groupby("build").agg(
    n=("player", "size"),
    star_seasons=("either40", lambda x: int((x >= 1).sum())),
    median_career_games=("career_games", "median"),
)

,n,star_seasons,median_career_games
build,,,
Mid-sized (under 191cm),25,1,55.0
Tall (191cm+),27,3,63.0


In [6]:
# Mid-sized slow starters who played on: the closest comparisons to Pedlar
played_on[played_on["height_cm"] < 191].sort_values(
    "career_games", ascending=False)[COLS].head(15).reset_index(drop=True)

,draft_year,pick,player,club,height_cm,first_5,games_after_5,career_games,squad,either40
0,2009,6,Gary Rohan,Sydney,189.0,9-9-4-5-16,161,204,0,0
1,2015,13,Matthew Kennedy,GWS,188.0,3-16-12-10-7,117,165,0,0
2,2014,20,Jayden Laverde,Essendon,190.0,9-9-5-7-10,124,164,0,0
3,2010,5,Jared Polec,Brisbane,188.0,3-12-1-24-5,103,148,0,0
4,2016,20,Isaac Cumming,GWS,183.0,0-2-5-3-24,88,122,0,0
5,2016,5,Will Setterfield,GWS,190.0,2-0-18-16-8,51,95,0,0
6,2015,6,Aaron Francis,Essendon,190.0,3-2-5-17-8,49,84,0,0
7,2015,17,Tom Doedee,Adelaide,187.0,0-0-20-1-9,53,83,0,0
8,2009,17,Daniel Menzel,Geelong,188.0,3-18-0-0-0,59,80,0,0
9,2011,5,Matt Buntine,GWS,189.0,6-3-7-11-12,28,67,0,0


## Summary

- **A slow start is a strong warning sign.** Of the 90 top-20 picks (2005–2018)
  with under 50 games in their first 5 seasons, 4% had a star season, against
  57% of those with 50 or more. None of the 38 who were gone by year 6 had one.
- **Of those who played on, 8% had a star season**, and three of the four were
  key-position talls: Harry McKay, Tom T. Lynch and Robbie Tarrant.
- **The only mid-sized one was Will Brodie**, and only just: 11 Brownlow votes
  in his one full season at Fremantle (2022), when 10 made the top 40. He's now
  at Port as well.
- **The best mid-sized careers were useful players, not stars:** Gary Rohan,
  Matt Kennedy, Jayden Laverde, Jared Polec and Isaac Cumming each played
  120–200 games.
- **For Pedlar**, history points to a solid, long-serving player as the good
  outcome, with a star ceiling very unlikely. That's still good value for moving
  from pick 32 to 34, where a typical pick produces a depth player anyway.

Sources for Pedlar's injuries and trade:
[AFL.com.au](https://www.afl.com.au/news/1628711/adelaide-crows-forward-luke-pedlar-traded-to-arch-rival-port-adelaide-power),
[Wikipedia](https://en.wikipedia.org/wiki/Luke_Pedlar).